# optimizer-state-tensor-buffers composite — cx4: allocate velocity buffer, then update it via copy_ (in-place)

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `optimizer-state-tensor-buffers`, `buffer-copy_-inplace`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "optimizer-state-tensor-buffers"
DD_ATOM_IDS = ["optimizer-state-tensor-buffers", "buffer-copy_-inplace"]
DD_SUBTOPICS = ["Optimizer: Per-param state buffers", "PyTorch: in-place buffer copy"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

When an optimizer maintains state buffers (e.g. SGD's velocity, Adam's first/second moments), the update rule MUST mutate those buffers IN PLACE. If you write `self.velocities[i] = momentum * v + g`, you've created a NEW tensor and rebound the list slot — fine for the next step, but breaks anyone who held a reference to the OLD buffer (e.g. a `.to(device)` call, a `state_dict` snapshot, or an external param-group view).

ARENA's convention (and PyTorch's): compute the new buffer value as a fresh tensor, then `buffer.copy_(new_value)` to write it back in place. `copy_` is the **broadcast-aware in-place copy** — it preserves the underlying storage of `buffer` while overwriting all of its elements with the source's values.

**The two atoms.**
- **optimizer-state-tensor-buffers** (atom A) — `self.velocities[i]` is a tensor allocated at `__init__` with the right shape/dtype/device.
- **buffer-copy_-inplace** (atom B) — `self.velocities[i].copy_(new_velocity)` writes the next-iteration value back into the SAME tensor.

**Anatomy of one momentum step.**
```python
for p, v in zip(self.params, self.velocities):
    if p.grad is None: continue
    g = p.grad
    new_v = self.momentum * v + g          # fresh tensor.
    v.copy_(new_v)                         # write into the SAME buffer (atom B).
    p.data -= self.lr * v                  # use the just-updated v.
```

Equivalent in-place idioms: `v.mul_(self.momentum).add_(g)` does the same thing without the intermediate `new_v`. We use `copy_` here because it's the most general pattern — Adam needs more than one step of arithmetic before writing the buffer back.

### Composite Exercise — allocate velocity buffer, then update it via copy_ (in-place)

**Atoms exercised together**: `optimizer-state-tensor-buffers`, `buffer-copy_-inplace`

Implement `cx4_make_sgdm_with_copy()` — return a class `SGDMomentumCopy`.

- `SGDMomentumCopy(params, lr, momentum=0.9)`:
  - `self.params = list(params)`
  - `self.lr = lr; self.momentum = momentum`
  - `self.velocities = [t.zeros_like(p) for p in self.params]` (atom A).
- `SGDMomentumCopy.step(self)`:
  - For each `(p, v)` in `zip(self.params, self.velocities)`:
    - If `p.grad is None`, skip.
    - Compute `new_v = self.momentum * v + p.grad` (FRESH tensor — NOT in-place).
    - Write back with `v.copy_(new_v)` (atom B — in-place into v).
    - Apply `p.data -= self.lr * v`.
  - Wrap in `t.no_grad()` (or decorator).

The test checks: (a) `self.velocities[i]` is the SAME tensor object after `.step()` (proves `copy_` was used, not reassignment); (b) the velocity VALUES are the correct momentum-update result; (c) the params updated correctly; (d) cross-check vs `torch.optim.SGD(..., momentum=0.9)`.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx4_make_sgdm_with_copy():
    """Return the SGDMomentumCopy class."""
    raise NotImplementedError

def _test_cx4():
    SGDMC = cx4_make_sgdm_with_copy()
    assert isinstance(SGDMC, type)

    # Case A: in-place copy_ — velocity tensor identity preserved across step().
    t.manual_seed(0)
    p = t.nn.Parameter(t.randn(3, 4))
    p.grad = t.ones_like(p)
    opt = SGDMC([p], lr=0.1, momentum=0.9)
    v_obj_before = opt.velocities[0]            # capture the tensor object.
    v_storage_id = v_obj_before.data_ptr()      # capture storage pointer.
    opt.step()
    assert opt.velocities[0] is v_obj_before, (
        'velocity tensor was REPLACED — must use buffer.copy_(new), not buffer = new'
    )
    assert opt.velocities[0].data_ptr() == v_storage_id, (
        'velocity storage changed — copy_ should preserve storage'
    )

    # Case B: velocity VALUES are correct after one step.
    # v0 = 0; v1 = 0.9*0 + grad = grad = ones.
    assert t.allclose(opt.velocities[0], t.ones_like(p)), (
        f'after step 1, velocity should equal grad (=ones); got {opt.velocities[0]}'
    )

    # Case C: second step — v2 = 0.9*v1 + grad = 0.9 + 1 = 1.9 (broadcast over the tensor).
    p.grad = t.ones_like(p)
    opt.step()
    assert t.allclose(opt.velocities[0], 1.9 * t.ones_like(p), atol=1e-6), (
        'after step 2, velocity should be 1.9 (= 0.9*1 + 1)'
    )

    # Case D: cross-check vs torch.optim.SGD with momentum on identical setup.
    t.manual_seed(1)
    w_init = t.randn(5, 2)
    p_mine = t.nn.Parameter(w_init.clone())
    p_ref = t.nn.Parameter(w_init.clone())
    opt_mine = SGDMC([p_mine], lr=0.05, momentum=0.9)
    opt_ref = t.optim.SGD([p_ref], lr=0.05, momentum=0.9)
    # Run 3 steps with identical gradients.
    for step_i in range(3):
        g = t.randn_like(p_mine) + step_i  # vary per step.
        p_mine.grad = g.clone()
        p_ref.grad = g.clone()
        opt_mine.step()
        opt_ref.step()
        assert t.allclose(p_mine.data, p_ref.data, atol=1e-6), (
            f'step {step_i}: my params diverge from torch.optim.SGD; '
            f'max err = {(p_mine.data - p_ref.data).abs().max().item()}'
        )
    _dd_passed.add('cx4')

_test_cx4()

<details><summary>Show solution — cx4</summary>

```python
def cx4_make_sgdm_with_copy():
    class SGDMomentumCopy:
        def __init__(self, params, lr, momentum=0.9):
            self.params = list(params)
            self.lr = lr
            self.momentum = momentum
            # Atom A (optimizer-state-tensor-buffers): one velocity per param.
            self.velocities = [t.zeros_like(p) for p in self.params]

        @t.no_grad()
        def step(self):
            for p, v in zip(self.params, self.velocities):
                if p.grad is None:
                    continue
                # Compute the new velocity as a FRESH tensor.
                new_v = self.momentum * v + p.grad
                # Atom B (buffer-copy_-inplace): write back into the SAME buffer.
                v.copy_(new_v)
                # Param update uses the just-updated v.
                p.data -= self.lr * v

    return SGDMomentumCopy
```

Why `copy_` and not `v[...] = new_v`? Both work for the basic case, but `copy_` is the idiom PyTorch uses internally (see `torch/optim/_functional.py`) because it's broadcast-aware and handles dtype/device casts safely. The mathematically-equivalent `v.mul_(self.momentum).add_(p.grad)` is faster (no intermediate `new_v` allocation) and is what PyTorch's fused SGD uses, but `copy_` reads more clearly when the new value involves multi-step arithmetic (think Adam's `m_hat = m / (1 - beta1**step)`).
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx4'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx4',
        'subtopics': ["Optimizer: Per-param state buffers", "PyTorch: in-place buffer copy"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()